In [1]:
import torch
import torch.nn as nn

z = torch.tensor([[2.0, 2.0, 2.0]], requires_grad=True)   # 세 클래스의 로짓이 같다
t = torch.tensor([2])                                      # 정답은 2번 클래스
nn.CrossEntropyLoss()(z, t).backward()
print(z.grad)

tensor([[ 0.3333,  0.3333, -0.6667]])


In [2]:
class M(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = torch.tensor([1.0], requires_grad=True)   # 일반 텐서
        self.b = nn.Parameter(torch.tensor([1.0]))

    def forward(self, x):
        return self.w * x + self.b

m = M()
m(torch.tensor([2.0])).sum().backward()       # 손실 = 2w + b
with torch.no_grad():
    for p in m.parameters():
        p -= 0.5 * p.grad
print(m.w.grad, m.w.item(), m.b.item())

tensor([2.]) 1.0 0.5


In [3]:
src = nn.Linear(1, 1)
dst = nn.Linear(1, 1)
out = dst.load_state_dict(src.state_dict())
print(type(out).__name__, torch.equal(dst.weight, src.weight))   # torch.equal: shape·값이 모두 같으면 True

_IncompatibleKeys True


# S1

In [4]:
import torch
import torch.nn as nn

x = torch.linspace(-3, 3, 100).unsqueeze(1)   # (100, 1)
y = 3 * x + 2                                  # (100, 1)
loss_fn = nn.MSELoss()
print(x.shape, y.shape)

torch.Size([100, 1]) torch.Size([100, 1])


In [5]:
torch.manual_seed(0)
model_a = nn.Linear(1, 1)          # 손으로 갱신할 모델
model_b = nn.Linear(1, 1)          # 옵티마이저로 갱신할 모델
r = model_b.load_state_dict(model_a.state_dict())   # a의 값을 b에 복사
print(r)
print(model_a.weight.item(), model_b.weight.item())

<All keys matched successfully>
-0.007486820220947266 -0.007486820220947266


In [6]:
lr = 0.1
for step in range(100):
    model_a.zero_grad()
    loss = loss_fn(model_a(x), y)
    loss.backward()
    with torch.no_grad():
        for p in model_a.parameters():
            p -= lr * p.grad
print(f"손 갱신   : w={model_a.weight.item():.6f}  b={model_a.bias.item():.6f}")

손 갱신   : w=3.000000  b=2.000000


In [7]:
opt = torch.optim.SGD(model_b.parameters(), lr=0.1)   # 루프 밖에서 한 번
for step in range(100):
    opt.zero_grad()
    loss = loss_fn(model_b(x), y)
    loss.backward()
    opt.step()
print(f"옵티마이저: w={model_b.weight.item():.6f}  b={model_b.bias.item():.6f}")

옵티마이저: w=3.000000  b=2.000000


In [8]:
for (name, pa), pb in zip(model_a.named_parameters(), model_b.parameters()):   # (이름, 파라미터) 쌍
    print(name, (pa - pb).abs().max().item())

weight 0.0
bias 0.0


In [9]:
group = opt.param_groups[0]
print(len(group["params"]), group["lr"])
print(group["params"][0] is model_b.weight, group["params"][1] is model_b.bias)
print(group["params"][0] is model_a.weight)

2 0.1
True True
False


In [10]:
import torch
print(hasattr(torch, "_utils"))
import torch._dynamo
print("dynamo OK")

True
dynamo OK


# S2


In [14]:
p = torch.zeros(1, requires_grad=True)
opt = torch.optim.SGD([p], lr=0.1, momentum=0.0)

for t in range(1, 4):
    before = p.item()
    opt.zero_grad()
    p.sum().backward()             # 손실 = p 그대로 → p.grad는 매번 1
    opt.step()
    print(t, f"이동 {before - p.item():.4f}")

1 이동 0.1000
2 이동 0.1000
3 이동 0.1000


In [15]:
print(opt.state[p])

{}


In [16]:
for t in range(4, 101):
    before = p.item()
    opt.zero_grad()
    p.sum().backward()
    opt.step()
print(f"100번째 이동 {before - p.item():.3f}   버퍼 {opt.state[p]['momentum_buffer'].item():.4f}")

KeyError: 'momentum_buffer'

In [17]:
p = torch.zeros(1, requires_grad=True)
opt = torch.optim.SGD([p], lr=0.1, momentum=0.9)
for t in range(1, 101):
    sign = 1.0 if t % 2 == 1 else -1.0      # +1, -1, +1, ...
    opt.zero_grad()
    (sign * p).sum().backward()             # p.grad = sign
    opt.step()
    if t >= 97:
        print(t, f"버퍼 {opt.state[p]['momentum_buffer'].item():+.4f}")

97 버퍼 +0.5263
98 버퍼 -0.5263
99 버퍼 +0.5263
100 버퍼 -0.5263


# S3

In [18]:
big   = torch.tensor([5.0], requires_grad=True)
small = torch.tensor([5.0], requires_grad=True)
opt = torch.optim.Adam([big, small], lr=0.1)

loss = 1000 * big.sum() + 0.001 * small.sum()   # big.grad = 1000, small.grad = 0.001
loss.backward()
opt.step()
print(f"big {5 - big.item():.4f}   small {5 - small.item():.4f}")

big 0.1000   small 0.1000


In [19]:
for name, q in (("big", big), ("small", small)):
    s = opt.state[q]
    print(f"{name:5s} t={s['step'].item():.0f}  m={s['exp_avg'].item():.4g}  v={s['exp_avg_sq'].item():.4g}")

big   t=1  m=100  v=1000
small t=1  m=0.0001  v=1e-09


In [20]:
s = opt.state[big]
m_hat = s["exp_avg"] / (1 - 0.9 ** s["step"])          # (7)
v_hat = s["exp_avg_sq"] / (1 - 0.999 ** s["step"])     # (7)
print(f"{(0.1 * m_hat / (v_hat.sqrt() + 1e-8)).item():.4f}")   # (8)의 이동 거리

0.1000


In [25]:
for Opt in (torch.optim.Adam, torch.optim.AdamW):
    q = torch.tensor([1.0], requires_grad=True)
    opt = Opt([q], lr=0.1, weight_decay=0.01)
    (0 * q).sum().backward()          # 손실이 q와 무관 → q.grad는 None이 아니라 0
    opt.step()
    print(f"{Opt.__name__:6s} q = {q.item():.4f}")

Adam   q = 0.9000
AdamW  q = 0.9990


# s4

In [26]:
import gc
dev = "cuda"

def state_mib(opt_cls, **kw):
    net = nn.Linear(4096, 4096).to(dev)
    opt = opt_cls(net.parameters(), **kw)
    net(torch.randn(8, 4096, device=dev)).sum().backward()   # .grad까지 만들어 둔 상태에서
    before = torch.cuda.memory_allocated()
    opt.step()                                               # 상태는 첫 step()에서 생긴다
    after = torch.cuda.memory_allocated()
    return (after - before) / 2**20

In [27]:
print(f"SGD        : {state_mib(torch.optim.SGD, lr=0.1):7.2f} MiB")
print(f"SGD + mom  : {state_mib(torch.optim.SGD, lr=0.1, momentum=0.9):7.2f} MiB")
print(f"Adam       : {state_mib(torch.optim.Adam, lr=1e-3):7.2f} MiB")
gc.collect(); torch.cuda.empty_cache()                       # 측정 뒤 정리

SGD        :    0.00 MiB
SGD + mom  :   64.02 MiB
Adam       :  128.03 MiB


# S5

In [39]:
def train(model, opt, x, y, steps=200):
    for step in range(steps):
        opt.zero_grad()

        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()

    return model.weight.item(), model.bias.item(), loss.item()

In [40]:
def wrong_train(model, opt, x, y, steps=200):
    for step in range(steps):
        opt.zero_grad()

        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()

    return model.weight.item(), model.bias.item(), loss.item()

In [35]:
torch.manual_seed(0)
model = nn.Linear(1, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
print("시작   :", model.weight.item(), model.bias.item())
print("200스텝:", train_wrong(model, opt, x, y))

시작   : -0.007486820220947266 0.5364435911178589
200스텝: (3.0, 1.999999761581421, 5.456968088664825e-14)


In [36]:
x10 = x * 10
y10 = 3 * x10 + 2                        # 정답 w=3, b=2는 그대로

for name, (xx, yy) in {"원래": (x, y), "x×10": (x10, y10)}.items():
    torch.manual_seed(0)
    model = nn.Linear(1, 1)
    loss_fn(model(xx), yy).backward()
    print(f"{name:5s} w.grad={model.weight.grad.item():10.2f}   b.grad={model.bias.grad.item():6.2f}")

원래    w.grad=    -18.41   b.grad= -2.93
x×10  w.grad=  -1840.95   b.grad= -2.93


In [41]:
runs = [
    ("원래", x,   y,   torch.optim.SGD,  dict(lr=0.1)),
    ("원래", x,   y,   torch.optim.Adam, dict(lr=0.1)),
    ("x×10", x10, y10, torch.optim.SGD,  dict(lr=0.1)),
    ("x×10", x10, y10, torch.optim.SGD,  dict(lr=0.003)),
    ("x×10", x10, y10, torch.optim.SGD,  dict(lr=0.003, momentum=0.9)),
    ("x×10", x10, y10, torch.optim.Adam, dict(lr=0.1)),
]
for i, (name, xx, yy, Opt, kw) in enumerate(runs, 1):
    torch.manual_seed(0)
    model = nn.Linear(1, 1)
    opt = Opt(model.parameters(), **kw)          # 모델을 만든 바로 다음 줄에서 만든다
    w, b, _ = train(model, opt, xx, yy)
    print(f"{i} {name:5s} {Opt.__name__:4s} {str(kw):32s} w={w:8.4f}  b={b:8.4f}")

1 원래    SGD  {'lr': 0.1}                      w=  3.0000  b=  2.0000
2 원래    Adam {'lr': 0.1}                      w=  3.0001  b=  2.0000
3 x×10  SGD  {'lr': 0.1}                      w=     nan  b=     nan
4 x×10  SGD  {'lr': 0.003}                    w=  3.0000  b=  1.5608
5 x×10  SGD  {'lr': 0.003, 'momentum': 0.9}   w=  2.9999  b=  2.0000
6 x×10  Adam {'lr': 0.1}                      w=  3.0001  b=  2.0000


In [42]:
print(1 / (x ** 2).mean().item(), 1 / (x10 ** 2).mean().item())

0.32673267943755574 0.003267326712928535


In [43]:
torch.manual_seed(0)
model = nn.Linear(1, 1)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
print(train(model, opt, x10, y10)[:2])

(0.18978454172611237, 0.7307376265525818)


# C3

In [56]:
class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.zeros(1, requires_grad=True))
        self.b = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return self.w * x + self.b

model = LinearModel()
opt = torch.optim.SGD(model.parameters(), lr=0.1)
for step in range(100):
    opt.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    opt.step()
print(f"loss={loss.item():.4f}  w={model.w.item():.4f}  b={model.b.item():.4f}")

loss=0.0000  w=3.0000  b=2.0000


In [49]:
print((model(x) - y).shape)
print(len(list(model.parameters())))

torch.Size([100, 1])
1
3.0000574588775635
1.9999639987945557


# 2

In [64]:
# 셀 A — 모델
torch.manual_seed(0)
model = nn.Linear(1, 1)

In [65]:
# 셀 B — 옵티마이저
opt = torch.optim.SGD(model.parameters(), lr=0.1)

In [66]:
# 셀 C — 학습 5스텝
for step in range(5):
    opt.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    opt.step()
    print(step, f"{loss.item():.6f}")

0 29.825108
1 5.535801
2 1.503974
3 0.655786
4 0.373551
